In [5]:
#If working locally, download the .py file and execute the following command in cmd to install required libraries
#pip install torch torchvision pillow opencv-python numpy

In [6]:
import torch
from torchvision import models, transforms
from PIL import Image
import cv2
import numpy as np
import urllib.request
import time
import os
import glob

In [7]:


# Ensure that you have the provided images in the same directory as the code
if not os.path.exists("./my_test_images"):
    raise FileNotFoundError(
        "\nPlease make sure you have the required input in this directory."
    )

if not os.path.exists("imagenet_classes.txt"):
    print("Downloading class labels...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt",
        "imagenet_classes.txt"
    )

with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]



In [8]:
#MODEL SETUP
print("Loading pre-trained ResNet18 model...")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.eval()

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def predict(image_path):

    img = Image.open(image_path).convert("RGB")
    input_tensor = transform(img).unsqueeze(0)

    # TODO 4: Add timing logic here to measure inference latency (in milliseconds)
    # Start timer
    start_time = time.perf_counter()

    with torch.no_grad():
        output = model(input_tensor)

    # End timer
    end_time = time.perf_counter()

    # Calculate latency in milliseconds (seconds * 1000)
    latency_ms = (end_time - start_time) * 1000.0

    probs = torch.nn.functional.softmax(output[0], dim=0)
    confidence, predicted_idx = torch.max(probs, 0)



    return categories[predicted_idx.item()], confidence.item(), latency_ms



Loading pre-trained ResNet18 model...


Implement the following functions:

1.   List item
2.   List item



In [9]:
def simulate_turbidity(img_array):
    """
    Simulate murky water using a blur effect.

    Args:
        img_array: OpenCV image array (BGR)

    Returns:
        Modified OpenCV image array with blur applied.
    """
    # Define the kernel size for the blur (must be odd numbers)
    # Larger numbers = heavier blur. 15x15 is a good starting point.
    kernel_size = (15, 15)

    # Apply Gaussian blur
    img_array= cv2.GaussianBlur(img_array, kernel_size, 0)

    return img_array

In [10]:
import cv2
import numpy as np

def simulate_color_shift(img_array):
    """
    Simulate depth color loss (attenuate the red channel).
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """


    # 2. Extract the Red channel (Index 2 in BGR) and convert to float32
    red_channel = img_array[:, :, 2].astype(np.float32)

    # 3. Apply a fixed attenuation (e.g., 0.3 leaves 30% of the red light)
    red_channel = red_channel * 0.3

    # 4. Clip values to stay within 0-255 and cast back to uint8
    img_array[:, :, 2] = np.clip(red_channel, 0, 255).astype(np.uint8)

    return img_array

In [11]:
def simulate_sensor_noise(img_array):
    """
    Simulate low-light digital camera noise.
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """
    # 1. Set noise parameters internally
    mean = 0
    std_dev = 25  # Higher numbers = heavier grain/noise

    # 2. Generate random Gaussian noise matching the image's exact dimensions
    noise = np.random.normal(mean, std_dev, img_array.shape)

    # 3. Convert image to float32 to safely add negative and positive noise
    # This prevents math errors (like 250 + 10 wrapping around to 4 in uint8)
    img_array= img_array.astype(np.float32) + noise

    # 4. Clip the results to ensure no pixel falls below 0 or goes above 255
    img_array = np.clip(img_array, 0, 255)

    # 5. Cast the final array back to standard 8-bit format
    return img_array.astype(np.uint8)

In [13]:
if __name__ == "__main__":

    # 1. Define the folder where your base images are stored
    image_directory = "./my_test_images"

    # 2. Grab all .png and .jpg files in that folder
    # Adjust the extensions if you are using .jpeg, .tiff, etc.
    image_paths = glob.glob(os.path.join(image_directory, "*.png"))
    image_paths.extend(glob.glob(os.path.join(image_directory, "*.jpg")))

    print("\nRUNNING VISION DIAGNOSTICS FOR MULTIPLE IMAGES")
    print("=" * 50)

    # 3. Loop through every image found in the directory
    for base_image in image_paths:
        print(f"\nEvaluating Image: {os.path.basename(base_image)}")

        img = cv2.imread(base_image)

        # Safety check in case the image is corrupted or unreadable
        if img is None:
            print(f"Error: Could not read {base_image}. Skipping...")
            continue

        # 4. Create dynamic filenames based on the original image name
        # This prevents test_turbid.jpg from overwriting itself on every loop
        base_name = os.path.basename(base_image).split('.')[0]

        turbid_path = f"test_turbid_{base_name}.jpg"
        colorshift_path = f"test_colorshift_{base_name}.jpg"
        noise_path = f"test_noise_{base_name}.jpg"

        # Generate and save the simulated conditions
        cv2.imwrite(turbid_path, simulate_turbidity(img.copy()))
        cv2.imwrite(colorshift_path, simulate_color_shift(img.copy()))
        cv2.imwrite(noise_path, simulate_sensor_noise(img.copy()))

        images_to_test = [
            ("Baseline (Clean)", base_image),
            ("Turbidity", turbid_path),
            ("Color Shift", colorshift_path),
            ("Sensor Noise", noise_path)
        ]

        # Run predictions for the current image's batch
        for condition_name, file_path in images_to_test:
            label, conf, latency = predict(file_path)
            print(f"Condition : {condition_name}")
            print(f"Prediction: {label}")
            print(f"Confidence: {conf:.4f}")
            print(f"Latency   : {latency:.2f} ms")
            print("-" * 30)

    print("\nALL DIAGNOSTICS COMPLETE")


RUNNING VISION DIAGNOSTICS FOR MULTIPLE IMAGES

Evaluating Image: set_f20_SESR.png
Condition : Baseline (Clean)
Prediction: hen-of-the-woods
Confidence: 0.2156
Latency   : 85.96 ms
------------------------------
Condition : Turbidity
Prediction: rock beauty
Confidence: 0.1679
Latency   : 11.89 ms
------------------------------
Condition : Color Shift
Prediction: coral reef
Confidence: 0.2957
Latency   : 9.09 ms
------------------------------
Condition : Sensor Noise
Prediction: coral reef
Confidence: 0.2287
Latency   : 10.84 ms
------------------------------

Evaluating Image: set_f46_SESR.png
Condition : Baseline (Clean)
Prediction: king crab
Confidence: 0.3312
Latency   : 10.47 ms
------------------------------
Condition : Turbidity
Prediction: barn spider
Confidence: 0.3702
Latency   : 9.77 ms
------------------------------
Condition : Color Shift
Prediction: king crab
Confidence: 0.6508
Latency   : 10.56 ms
------------------------------
Condition : Sensor Noise
Prediction: king c

In [14]:
import os
print("My images are being saved to:", os.getcwd())

My images are being saved to: C:\Users\airfo\PycharmProjects\first
